# Optimization At Scale

Large-model training bottlenecks are often optimizer bottlenecks disguised as model bottlenecks. AdamW keeps two running moments per parameter, gradient accumulation changes the meaning of one optimizer step, and mixed precision changes storage without automatically removing optimizer-state overhead. This notebook keeps those three pieces explicit with small deterministic examples and ties them back to the systems literature on decoupled weight decay, ZeRO, Megatron-LM, and width-aware transfer [@decoupled_weight_decay_regularization_2017; @zero_2019; @megatron_lm_2019; @tensor_programs_v_2022].

## Motivation

At small scale it is easy to think of optimization as a local algorithmic detail: pick AdamW, pick a learning rate, and start training. At scale that story breaks. Optimizer moments and related training tensors can dominate the tracked memory budget, accumulation changes how many optimizer steps happen per epoch, and mixed precision only helps if we account for which tensors stay in fp32.

The practical goal here is narrower than "train a giant model": reproduce one AdamW update exactly, make accumulation accounting explicit, and turn the tracked AdamW training state into a simple byte budget that can be checked before launching anything expensive.

## Hypothesis

If we implement a minimal reference AdamW update with the same semantics PyTorch uses for the non-AMSGrad path, then it should match `torch.optim.AdamW` on a controlled tiny tensor to floating-point tolerance. If we average equal-size microbatch gradients correctly, then accumulated gradients should equal the full-batch gradient. Finally, if we count parameters, gradients, moments, and optional master weights separately, then the resulting memory budget should show why optimizer state remains a first-order systems concern even when parameter storage is reduced.

## Baseline

The misleading baseline is the casual optimization story:

1. treat weight decay as if it were just another gradient term;
2. treat gradient accumulation as if it changed memory but not schedule accounting;
3. treat lower-precision parameters as if they alone determine optimizer memory.

All three shortcuts can mislead scaling decisions. AdamW decouples weight decay from adaptive moments [@decoupled_weight_decay_regularization_2017], accumulation changes the number of optimizer updates per token budget, and distributed systems papers such as ZeRO and Megatron-LM exist largely because the full training state does not fit cleanly into the naive baseline [@zero_2019; @megatron_lm_2019].

## Metric

We track four diagnostics:

- maximum absolute parameter mismatch between the reference update and `torch.optim.AdamW`;
- maximum absolute mismatch between the full-batch gradient and the accumulated microbatch gradient;
- effective batch size, tokens per update, and optimizer steps per epoch under accumulation;
- per-component optimizer memory totals in bytes and GiB.

A schedule sanity check is also included: the notebook prints a tiny warmup-plus-cosine schedule under two accumulation settings to make step-indexing visible.

## Mathematical derivation

For AdamW with parameters `theta`, gradient `g_t`, first moment `m_t`, second moment `v_t`, learning rate `eta`, decay `lambda`, and betas `(beta_1, beta_2)`, the non-AMSGrad update is

`m_t = beta_1 m_{t-1} + (1 - beta_1) g_t`

`v_t = beta_2 v_{t-1} + (1 - beta_2) g_t^2`

`m_hat_t = m_t / (1 - beta_1^t)`

`v_hat_t = v_t / (1 - beta_2^t)`

`theta_t = (1 - eta lambda) theta_{t-1} - eta m_hat_t / (sqrt(v_hat_t) + eps)`

The decoupling matters: weight decay multiplies the parameter directly instead of being absorbed into the adaptive gradient statistics.

If microbatch `i` contributes mean gradient `g_i` over `n_i` examples, then the full-batch mean is

`g_acc = (sum_i n_i g_i) / (sum_i n_i)`.

For equal-sized microbatches this reduces to a simple average. The effective batch size per optimizer update is

`B_eff = B_micro * K * D`,

where `B_micro` is microbatch size, `K` is accumulation steps, and `D` is the data-parallel replica count.

A minimal tracked training-state budget for AdamW is

`M = N (b_param + b_grad + b_m1 + b_m2 + 1_master b_master)`,

where `N` is parameter count and the `b_*` terms are bytes per parameter tensor, gradient tensor, first moment, second moment, and optional fp32 master copy. Lowering only `b_param` and `b_grad` does not remove the moment terms.

For schedules we use a step-indexed warmup and cosine decay. The important bookkeeping fact is conceptual, not fancy: the step counter is the optimizer-step index, not the microbatch index. Changing accumulation without recomputing total optimizer steps shifts the schedule.

In [ ]:
from pathlib import Path
import sys
import math

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from llm_from_scratch.research.optimization import (
    accumulate_mean_gradients,
    adamw_reference_step,
    estimate_optimizer_memory,
    gradient_accumulation_stats,
)

torch.manual_seed(0)
torch.set_printoptions(precision=10, sci_mode=False)


def gib(num_bytes: int) -> float:
    return num_bytes / (2 ** 30)


def linear_warmup_cosine(step: int, *, warmup_steps: int, total_steps: int, base_lr: float) -> float:
    if not 1 <= step <= total_steps:
        raise ValueError('step must lie in [1, total_steps]')
    if warmup_steps <= 0:
        raise ValueError('warmup_steps must be positive')
    if total_steps < warmup_steps:
        raise ValueError('total_steps must be at least warmup_steps')
    if step <= warmup_steps:
        return base_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * base_lr * (1.0 + math.cos(math.pi * progress))

## PyTorch implementation

The module surface is intentionally small:

- `adamw_reference_step()` mirrors the standard non-AMSGrad PyTorch AdamW update for one tensor;
- `accumulate_mean_gradients()` combines microbatch gradients deterministically, with optional sample-count weighting;
- `gradient_accumulation_stats()` turns microbatch settings into effective-batch and optimizer-step counts;
- `estimate_optimizer_memory()` returns a byte breakdown for the tracked training state: parameters, gradients, moments, and optional master weights.

We start with a tiny tensor so the AdamW comparison is exact enough to be educational rather than approximate.

In [ ]:
parameter = torch.tensor([1.25, -0.75, 0.5], dtype=torch.float64)
gradients = [
    torch.tensor([0.1, -0.2, 0.05], dtype=torch.float64),
    torch.tensor([-0.3, 0.15, 0.4], dtype=torch.float64),
]
kwargs = {
    'lr': 1e-2,
    'betas': (0.9, 0.95),
    'eps': 1e-8,
    'weight_decay': 0.1,
}

actual_parameter = torch.nn.Parameter(parameter.clone())
optimizer = torch.optim.AdamW([actual_parameter], **kwargs)

reference_parameter = parameter.clone()
reference_state = None
for gradient in gradients:
    optimizer.zero_grad()
    actual_parameter.grad = gradient.clone()
    optimizer.step()
    reference_parameter, reference_state = adamw_reference_step(
        reference_parameter,
        gradient,
        state=reference_state,
        **kwargs,
    )

torch_state = optimizer.state[actual_parameter]
adamw_parameter_error = float((reference_parameter - actual_parameter.detach()).abs().max().item())
adamw_moment_error = float((reference_state.exp_avg - torch_state['exp_avg']).abs().max().item())
adamw_second_moment_error = float((reference_state.exp_avg_sq - torch_state['exp_avg_sq']).abs().max().item())

print('reference parameter:', reference_parameter)
print('torch parameter    :', actual_parameter.detach())
print('max |parameter diff|:', adamw_parameter_error)
print('max |exp_avg diff|  :', adamw_moment_error)
print('max |exp_avg_sq diff|:', adamw_second_moment_error)
print('step:', reference_state.step)

In [ ]:
features = torch.tensor(
    [
        [1.0, -1.0],
        [0.5, 0.25],
        [-0.75, 1.5],
        [1.25, 0.5],
    ],
    dtype=torch.float64,
)
targets = torch.tensor([[0.1], [0.2], [-0.3], [0.4]], dtype=torch.float64)
weight = torch.tensor([[0.4, -0.2]], dtype=torch.float64, requires_grad=True)

full_prediction = features @ weight.t()
full_loss = torch.mean((full_prediction - targets) ** 2)
full_gradient = torch.autograd.grad(full_loss, weight)[0]

micro_gradients = []
sample_counts = []
for micro_features, micro_targets in zip(features.chunk(2), targets.chunk(2), strict=True):
    micro_weight = weight.detach().clone().requires_grad_(True)
    micro_prediction = micro_features @ micro_weight.t()
    micro_loss = torch.mean((micro_prediction - micro_targets) ** 2)
    micro_gradients.append(torch.autograd.grad(micro_loss, micro_weight)[0])
    sample_counts.append(int(micro_features.shape[0]))

accumulated_gradient = accumulate_mean_gradients(micro_gradients, sample_counts=sample_counts)
accumulation_stats = gradient_accumulation_stats(
    micro_batch_size=2,
    accumulation_steps=2,
    data_parallel_size=4,
    sequence_length=128,
    micro_batches_per_epoch=18,
)

print('full-batch gradient:')
print(full_gradient)
print('accumulated gradient:')
print(accumulated_gradient)
print('effective batch size:', accumulation_stats.effective_batch_size)
print('tokens per update   :', accumulation_stats.tokens_per_update)
print('optimizer steps/epoch:', accumulation_stats.optimizer_steps_per_epoch)
print('remainder microbatches:', accumulation_stats.remainder_micro_batches)

## Numerical checks

The checks below are the notebook's contract tests in miniature: AdamW must agree with PyTorch on the controlled example, accumulated gradients must equal the full-batch gradient, and the memory estimator must reproduce a hand-computed total. The last comparison also makes a mixed-precision caveat concrete: bf16 parameters do not guarantee lower total optimizer memory if moments and master weights stay in fp32.

In [ ]:
fp32_adamw = estimate_optimizer_memory(
    num_parameters=1_000_000_000,
    parameter_bytes=4,
    gradient_bytes=4,
    state_bytes=4,
    use_master_weights=False,
)
mixed_precision_adamw = estimate_optimizer_memory(
    num_parameters=1_000_000_000,
    parameter_bytes=2,
    gradient_bytes=2,
    state_bytes=4,
    use_master_weights=True,
    master_weight_bytes=4,
)

gradient_error = float((accumulated_gradient - full_gradient).abs().max().item())

assert adamw_parameter_error < 1e-12
assert adamw_moment_error < 1e-12
assert adamw_second_moment_error < 1e-12
assert gradient_error < 1e-12
assert fp32_adamw.total_bytes == 16_000_000_000
assert mixed_precision_adamw.total_bytes == 16_000_000_000

print('AdamW parity check passed with max parameter error:', adamw_parameter_error)
print('Accumulated gradient check passed with max error  :', gradient_error)
print('fp32 AdamW total memory      :', f'{gib(fp32_adamw.total_bytes):.3f} GiB')
print('bf16 params/grads + fp32 state + master weights:', f'{gib(mixed_precision_adamw.total_bytes):.3f} GiB')
print('tracked training-state breakdown (bytes):')
print('  fp32   ->', fp32_adamw)
print('  mixed  ->', mixed_precision_adamw)

## Ablations

Two small ablations make the systems point sharper.

First, schedule indexing: if accumulation changes from 1 to 4, the epoch now contains four times fewer optimizer steps. Reusing the old step schedule is wrong even if the token budget is unchanged.

Second, memory breakdown: a mixed-precision recipe can reduce parameter and gradient storage while leaving optimizer moments dominant. That is one reason systems papers such as ZeRO matter, and one reason hyperparameter transfer across widths needs care rather than folklore [@zero_2019; @tensor_programs_v_2022].

In [ ]:
no_accumulation = gradient_accumulation_stats(
    micro_batch_size=8,
    accumulation_steps=1,
    data_parallel_size=1,
    sequence_length=2048,
    micro_batches_per_epoch=16,
)
with_accumulation = gradient_accumulation_stats(
    micro_batch_size=8,
    accumulation_steps=4,
    data_parallel_size=1,
    sequence_length=2048,
    micro_batches_per_epoch=16,
)

recomputed_schedule = [
    round(linear_warmup_cosine(step, warmup_steps=1, total_steps=with_accumulation.optimizer_steps_per_epoch, base_lr=3e-4), 7)
    for step in range(1, with_accumulation.optimizer_steps_per_epoch + 1)
]
wrong_if_old_steps_reused = [
    round(linear_warmup_cosine(step, warmup_steps=4, total_steps=no_accumulation.optimizer_steps_per_epoch, base_lr=3e-4), 7)
    for step in range(1, with_accumulation.optimizer_steps_per_epoch + 1)
]

illustrative_eight_way_moment_sharding = (
    mixed_precision_adamw.parameter_bytes
    + mixed_precision_adamw.gradient_bytes
    + mixed_precision_adamw.master_weights_bytes
    + (mixed_precision_adamw.first_moment_bytes + mixed_precision_adamw.second_moment_bytes) // 8
)

print('optimizer steps per epoch without accumulation:', no_accumulation.optimizer_steps_per_epoch)
print('optimizer steps per epoch with accumulation   :', with_accumulation.optimizer_steps_per_epoch)
print('recomputed 4-step schedule                    :', recomputed_schedule)
print('wrong first 4 steps if the 16-step schedule were reused:', wrong_if_old_steps_reused)
print('')
print('illustrative 1B-parameter memory comparison')
print('  fp32 AdamW                                 :', f'{gib(fp32_adamw.total_bytes):.3f} GiB')
print('  mixed precision with master weights         :', f'{gib(mixed_precision_adamw.total_bytes):.3f} GiB')
print('  mixed precision with 8-way moment sharding  :', f'{gib(illustrative_eight_way_moment_sharding):.3f} GiB')

## Assumptions

- The AdamW comparison targets the standard non-AMSGrad PyTorch path on CPU tensors.
- The gradient-accumulation example uses mean-reduced losses and equal-sized microbatches; weighted averaging is included for uneven cases.
- Memory accounting is a tensor-storage budget, not a full wall-clock or communication model.
- The warmup-plus-cosine schedule is a didactic step-indexed example, not a claim that one schedule is universally best.
- Mixed-precision caveats are discussed analytically here; the notebook does not require GPU kernels, fused optimizers, or dynamic loss scaling.

## Risks

A clean byte budget can still understate real systems difficulty. Communication, activation memory, fragmentation, optimizer fusion, and checkpointing policy all matter in practice. Likewise, matching one AdamW step does not prove that a large distributed training stack is correct end to end.

The mixed-precision discussion here is intentionally conservative. Many real training stacks keep moments in fp32 and may keep fp32 master weights, but exact storage choices depend on the implementation. ZeRO and Megatron-LM solve broader distributed systems problems than this notebook models [@zero_2019; @megatron_lm_2019]. MuP is included as a conceptual warning that optimizer hyperparameters need not transfer naively across width changes [@tensor_programs_v_2022].

## Failure criteria

Treat the notebook as failed if any of the following happen:

- the reference AdamW update disagrees with `torch.optim.AdamW` beyond floating-point tolerance on the controlled example;
- accumulated gradients differ from the full-batch gradient on the deterministic toy regression;
- the memory estimator cannot reproduce the hand-counted totals for the demonstrated precision settings;
- the schedule ablation fails to show that accumulation changes optimizer-step indexing;
- the notebook implies that mixed precision alone eliminates optimizer-state bottlenecks.

## Exercises

Work the companion exercise set in `exercises/research/24_optimization_at_scale.md` before reading the solutions.

Suggested prompts:

1. Re-derive the AdamW update and explain exactly where decoupled weight decay enters.
2. Compute the effective batch size and tokens per update for a new accumulation setting.
3. Compare two precision recipes and identify which tensor class dominates memory.
4. Explain why a step-indexed schedule must be recomputed after changing accumulation.
5. State one reason MuP is a warning against naive optimizer hyperparameter transfer across width.

## References

- Loshchilov and Hutter, *Decoupled Weight Decay Regularization* [@decoupled_weight_decay_regularization_2017]
- Rajbhandari et al., *ZeRO: Memory Optimizations Toward Training Trillion Parameter Models* [@zero_2019]
- Shoeybi et al., *Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism* [@megatron_lm_2019]
- Yang et al., *Tensor Programs V: Tuning Large Neural Networks via Zero-Shot Hyperparameter Transfer* [@tensor_programs_v_2022]